In [1]:
from tensorflow import keras

I0000 00:00:1783340242.737110   51522 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783340242.790725   51522 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783340244.306960   51522 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

In [3]:
import argparse
import numpy as np
import pandas as pd

In [4]:
X_train[0].shape

(28, 28)

In [5]:
X_train = X_train.astype(np.float32)/255.0
X_test = X_test.astype(np.float32)/255.0

In [6]:
# Transpose so each COLUMN is one training example (784, m).
X_train = X_train.T
X_test = X_test.T

In [14]:
y_train

array([5, 0, 4, ..., 5, 6, 8], shape=(60000,), dtype=uint8)

In [13]:

"""Convert integer labels (m,) into one-hot columns (num_classes, m)."""
one_hot_y =  np.zeros((y_train.size, 10))  # --> y_train.size of rows and 10 cols
one_hot_y.shape

(60000, 10)

In [19]:
one_hot_y[np.arange(y_train.size), y_train] = 1

In [23]:
one_hot_y = one_hot_y.T

In [ ]:
one_hot_y[0]

array([0., 1., 0., ..., 0., 0., 0.], shape=(60000,))

In [ ]:
# Params initialization
def init_params(input_size=784, hidden_size=128, output_size=10):
    """
    He initialization: scale weights by sqrt(2/fan_in). This keeps the
    variance of activations roughly constant across layers, which matters
    a lot when using ReLU (prevents vanishing/exploding activations).
    """
    W1 = np.random.randn(hidden_size, input_size) * np.sqrt(2.0/input_size)
    b1 = np.zeros((hidden_size, 1))
    W2 = np.random.randn(output_size, hidden_size) * np.sqrt(2.0/hidden_size)
    b2 = np.zeros((output_size, 1))

    return W1, b1, W2, b2

In [30]:
# activation functions
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z>0).astype(float)

def softmax(Z):
    Z_shifted = Z - np.max(Z, axis=0, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z/np.sum(exp_Z, axis=0, keepdims=True)

In [31]:
# forward propagation
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = relu(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

In [32]:
# Backward propagation
def backward(Z1, A1, A2, W2, X, Y, Y_ohe, m):
    # Gradient of cross-entropy loss w.r.t. Z2 simplifies beautifully
    # when paired with softmax: dZ2 = A2 - Y
    dZ2 = A2 - Y_ohe
    dW2 = (1/m)* dZ2.dot(A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    dZ1 = W2.T.dot(dZ2) * relu_derivative(Z1)
    dW1 = (1/m) * dZ1.dot(X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2

In [33]:
# Params update
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, lr):
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    return W1, b1, W2, b2

In [34]:
# Loss and accuracy
def compute_loss(A2, Y_ohe):
    """Categorical cross-entropy loss, averaged over the batch."""
    m = Y_ohe.shape[1]
    eps = 1e-8
    return -np.sum(Y_ohe * np.log(A2 + eps))/m

def get_predicted(A2):
    return np.argmax(A2, axis=0)

def get_accuracy(predictions, labels):
    return np.sum(predictions == labels)/labels